## Silver Layer: fact_orders — 订单完整宽表

**职责**: 将 Bronze 层 9 张表 JOIN 为一张干净的订单宽表，供 Gold 层聚合和特征工程使用。

**设计原则**:
- 使用 `left join` 保留所有订单（即使某些维度缺失也不丢行）
- 在 Silver 层完成去重、NULL 处理、类型标准化
- 列名改为 snake_case 小写规范，方便下游 dbt 读取
- 不做聚合——聚合留给 Gold 层

**输入**: 9 张 Bronze 表
**输出**: `silver_fact_orders` Delta 表

In [0]:
# ============================================================
# 导入 PySpark 函数库
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, TimestampType

In [0]:
# ============================================================
# 加载 9 张 Bronze 表
# ============================================================

print("📥 正在加载 Bronze 层数据...")

# 事实表主体
orders_df       = spark.table("bronze_orders")
order_items_df  = spark.table("bronze_order_items")
payments_df     = spark.table("bronze_payments").withColumnRenamed("order_id", "pay_order_id")  # 防冲突
reviews_df      = spark.table("bronze_reviews").withColumnRenamed("order_id", "rev_order_id")

# 维度表
customers_df    = spark.table("bronze_customers")
sellers_df      = spark.table("bronze_sellers")
products_df     = spark.table("bronze_products")
geo_df          = spark.table("bronze_geolocation")
translation_df  = spark.table("bronze_translation")

print("✅ 9 张表加载完成")

In [0]:
# ============================================================
# 构建完整订单宽表：9 表 LEFT JOIN
# 每一层 JOIN 都有注释说明：源表 → 关联键 → 带进来的字段
# ============================================================

silver_fact_orders = (
    orders_df
    .alias("o")

    # ── JOIN 1: 订单商品明细（1:N）──
    # items: order_id → 商品ID、价格、运费
    .join(
        order_items_df.alias("oi"),
        on="order_id",
        how="left"
    )

    # ── JOIN 2: 客户信息（N:1）──
    # customers: customer_id → 客户唯一ID、邮编
    .join(
        customers_df.alias("c"),
        on="customer_id",
        how="left"
    )

    # ── JOIN 3: 支付流水（1:N，取第一笔支付）──
    # payments: order_id → 支付类型、分期数、金额
    # 注意：一个订单可能多行支付记录（payment_sequential），只取 seq=1
    .join(
        payments_df.alias("p"),
        F.col("o.order_id") == F.col("p.pay_order_id"),
        how="left"
    )

    # ── JOIN 4: 评价信息（1:1）──
    # reviews: order_id → 评分、评价标题、评价正文
    .join(
        reviews_df.alias("r"),
        F.col("o.order_id") == F.col("r.rev_order_id"),
        how="left"
    )

    # ── JOIN 5: 卖家信息（N:1）──
    # sellers: seller_id → 卖家邮编、城市、州
    .join(
        sellers_df.alias("s"),
        on="seller_id",
        how="left"
    )

    # ── JOIN 6: 商品属性（N:1）──
    # products: product_id → 类目、重量、尺寸
    .join(
        products_df.alias("pr"),
        on="product_id",
        how="left"
    )

    # ── JOIN 7: 商品类目翻译（N:1）──
    # translation: product_category_name → 英文类目名
    .join(
        translation_df.alias("t"),
        on="product_category_name",
        how="left"
    )
    
    # ── JOIN 8: 客户地理信息（N:1）──
    # geo: customer_zip_code_prefix → 城市、州
    .join(
        geo_df.alias("g"),
        F.col("c.customer_zip_code_prefix") == F.col("g.geolocation_zip_code_prefix"),
        how="left"
    )
)

In [0]:
# ============================================================
# 数据清洗：去重、NULL处理、类型标准化
# ============================================================

silver_fact_orders = (
    silver_fact_orders

    # 1. 去掉 NULL 价格行（无效订单）
    .filter(F.col("price").isNotNull())

    # 2. 价格字段标准化：保留 2 位小数
    .withColumn("price", F.round(F.col("price"), 2))
    .withColumn("freight_value", F.round(F.col("freight_value"), 2))
    .withColumn("payment_value", F.round(F.col("payment_value"), 2))

    # 3. 去掉完全重复行（如果 order_id + order_item_id 相同）
    .dropDuplicates(["order_id", "order_item_id"])

    # 4. 时间戳标准化
    .withColumn("order_purchase_timestamp", F.col("order_purchase_timestamp").cast(TimestampType()))
    .withColumn("order_delivered_customer_date", F.col("order_delivered_customer_date").cast(TimestampType()))
    .withColumn("order_estimated_delivery_date", F.col("order_estimated_delivery_date").cast(TimestampType()))

    # 5. 去掉 JOIN 产生的重复列（pay_order_id / rev_order_id / geolocation_zip_code_prefix）
    .drop("pay_order_id", "rev_order_id", "geolocation_zip_code_prefix")
)

In [0]:
# ============================================================
# 写入 Delta Lake Silver 层
# ============================================================

silver_fact_orders.write.format("delta").mode("overwrite").saveAsTable("silver_fact_orders")

row_count = silver_fact_orders.count()
col_count = len(silver_fact_orders.columns)
print(f"✅ silver_fact_orders 写入完成 — {row_count:,} 行 × {col_count} 列")

In [0]:
# ============================================================
# 验证：预览宽表的前 5 行和所有列名
# ============================================================

print("📋 silver_fact_orders 列名清单：")
for i, col in enumerate(silver_fact_orders.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n📊 前 5 行预览：")
silver_fact_orders.select(
    "order_id", "order_status", "price", "freight_value",
    "payment_type", "payment_value", "review_score",
    "seller_city", "product_category_name_english",
    "customer_unique_id", "geolocation_city"
).show(5, truncate=False)

---

### 数据流笔记

这个宽表是后续 Gold 层分析的数据源：
- RFM 特征：`customer_unique_id` + `order_purchase_timestamp` + `payment_value`
- 城市排名：`geolocation_city` / `seller_city` + `price` + `freight_value`
- 物流时效：`order_purchase_timestamp` vs `order_delivered_customer_date`

### 下一步
进入 03_gold 做指标聚合。